<a href="https://colab.research.google.com/github/MalshanRuchira/NorthStar-Analytics-Project/blob/main/NorthStar_Main_Analysis_MongoDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
from pymongo import MongoClient
import json

client = MongoClient('localhost', 27017)
db = client['NorthStarApp']
collection = db['CustomerTracking']
collection.drop()

orders_df = pd.read_csv('https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/orders.csv')
deliveries_df = pd.read_csv('https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/deliveries.csv')
complaints_df = pd.read_csv('https://raw.githubusercontent.com/MalshanRuchira/NorthStar-Analytics-Project/refs/heads/main/complaints.csv')

mongo_documents = []
for index, order in orders_df.iterrows():
    order_id = order['order_id']
    delivery = deliveries_df[deliveries_df['order_id'] == order_id]
    complaint = complaints_df[complaints_df['order_id'] == order_id]

    doc = {
        "_id": order_id,
        "customer_id": order['customer_id'],
        "service_type": order['service_type'],
        "financials": {"order_value": float(order['order_value'])},
        "delivery_tracking": {},
        "customer_support": {}
    }

    if not delivery.empty:
        doc["delivery_tracking"] = {
            "status": delivery.iloc[0]['delivery_status'],
            "driver_id": delivery.iloc[0]['driver_id']
        }
    if not complaint.empty:
        doc["customer_support"] = {
            "issue_type": complaint.iloc[0]['complaint_type'],
            "resolved_in_days": float(complaint.iloc[0]['resolution_days'])
        }
    mongo_documents.append(doc)

collection.insert_many(mongo_documents)
print("CREATE: Inserted documents successfully.")

print("\n--- SCHEMA EXAMPLE (One Full Document) ---")
print(json.dumps(collection.find_one(), indent=4))

print("\n--- UPDATE OPERATION ---")
sample_id = collection.find_one()["_id"]
collection.update_one(
    {"_id": sample_id},
    {"$set": {"delivery_tracking.status": "Expedited_Delivery"}}
)
print(f"Updated delivery status for order: {sample_id}")


print("\n--- DELETE OPERATION ---")
collection.insert_one({"_id": "TEST_ORD_999", "junk_data": True})
collection.delete_one({"_id": "TEST_ORD_999"})
print("Deleted dummy document TEST_ORD_999.")

print("\n--- AGGREGATION 1 ---")
agg1 = collection.aggregate([
    {"$group": {"_id": "$service_type", "average_value": {"$avg": "$financials.order_value"}}}
])
for result in agg1:
    print(result)


print("\n--- AGGREGATION 2 ---")
agg2 = collection.aggregate([
    {"$match": {"customer_support.issue_type": {"$exists": True}}},
    {"$group": {"_id": "$customer_support.issue_type", "total_complaints": {"$sum": 1}}},
    {"$sort": {"total_complaints": -1}}
])
for result in agg2:
    print(result)

CREATE: Inserted documents successfully.

--- SCHEMA EXAMPLE (One Full Document) ---
{
    "_id": "O00001",
    "customer_id": "C0292",
    "service_type": "Passenger",
    "financials": {
        "order_value": 126.65
    },
    "delivery_tracking": {
        "status": "OnTime",
        "driver_id": "D047"
    },
    "customer_support": {}
}

--- UPDATE OPERATION ---
Updated delivery status for order: O00001

--- DELETE OPERATION ---
Deleted dummy document TEST_ORD_999.

--- AGGREGATION 1 ---
{'_id': 'Parcel', 'average_value': 87.61564935064935}
{'_id': 'Retail', 'average_value': 90.01367003367004}
{'_id': 'Passenger', 'average_value': 96.07363636363637}
{'_id': 'Medical', 'average_value': 87.13618705035971}
{'_id': 'Business', 'average_value': 92.2450303030303}

--- AGGREGATION 2 ---
{'_id': 'Delay', 'total_complaints': 88}
{'_id': 'MissedPickup', 'total_complaints': 59}
{'_id': 'AppIssue', 'total_complaints': 48}
{'_id': 'DriverBehaviour', 'total_complaints': 46}
{'_id': 'SupportExp

In [6]:
target_customer = "CUS-8392"


print("--- BEFORE INDEX CREATION ---")
unoptimized = collection.find({"customer_id": target_customer}).explain()["executionStats"]
print(f"Execution Time (ms): {unoptimized['executionTimeMillis']}")
print(f"Total Docs Examined: {unoptimized['totalDocsExamined']}")
print(f"Total Docs Returned: {unoptimized['nReturned']}")


print("\n--- CREATING INDEX ---")
collection.create_index("customer_id")
print("Index created on 'customer_id' field.")


print("\n--- AFTER INDEX CREATION ---")
optimized = collection.find({"customer_id": target_customer}).explain()["executionStats"]
print(f"Execution Time (ms): {optimized['executionTimeMillis']}")
print(f"Total Docs Examined: {optimized['totalDocsExamined']}")
print(f"Total Docs Returned: {optimized['nReturned']}")

--- BEFORE INDEX CREATION ---
Execution Time (ms): 1
Total Docs Examined: 1250
Total Docs Returned: 0

--- CREATING INDEX ---
Index created on 'customer_id' field.

--- AFTER INDEX CREATION ---
Execution Time (ms): 1
Total Docs Examined: 0
Total Docs Returned: 0
